In [1]:
from alerce.core import Alerce
from astropy.table import vstack, Table
import json
import matplotlib.pyplot as plt
import pyvo as vo
import requests
import sqlalchemy as sa
import sys
import numpy as np
from astropy.coordinates import SkyCoord
import george
import scipy.optimize as op
import emcee
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import george
import scipy.optimize as op
import emcee

In [2]:
import utils

In [3]:
#set SN type for processing
curve_type = "Ia" #options: Ia, Ib, Ic, II

In [4]:
#read in light curves
with open(f"../Data/{curve_type}.json", 'r') as f:
    light_curves = json.load(f)

In [14]:
#intialize number of curves to process
max_curves = None #None will run all
dic_curves = []

#iterate through light curves
for light in light_curves[:max_curves]:

    #filter out curves with less than 11 detections
    if len(light) > 10:

        #fetch time, psuedo flux, and psuedo flux error
        time, mag_flipped, error = utils.decompose_curve(light)

        #check for empty curve
        if len(mag_flipped) > 0:
            
            #fit line to curve
            pred, pred_var, x_fit = utils.fit_curve(time, mag_flipped, error, mode="george")

            #find peak
            peak = np.max(pred)

            #find rise time
            rise_time = utils.get_rise_time(pred, x_fit, peak, mode="slope")
            #print(rise_time)

            #find fall time
            fall_time = utils.get_fall_time(pred, x_fit, peak, mode="slope")

            #check for rise/fall calculation error
            if rise_time < 0 or fall_time < 0:
                rise_time = utils.get_rise_time(pred, x_fit, peak, mode="basic")
                fall_time = utils.get_fall_time(pred, x_fit, peak, mode="basic")

            #add curve to dictionary and append to list
            error = np.pow(pred_var, 0.5)    
            dic = {"object": light[0]["oid"], "type": f"SN{curve_type}", "mjd": x_fit, "mag": pred, 
                   "error": error, "peak": peak, "rise": rise_time, "fall": fall_time}
            dic_curves.append(dic)

            #plot fit for checking fit
            #utils.plot_curve_fit(time, mag_flipped, x_fit, pred, light[0]["oid"], curve_type, peak, rise_time, fall_time)

#combine curves into table
table_curves = vstack(dic_curves)

#Ia table larger than github allows, break into two tables, otherwise save table
if curve_type == "Ia":
    table_curves1 = table_curves[:len(table_curves)//2]
    table_curves2 = table_curves[len(table_curves)//2:]
    table_curves1.write(f"../Data/SN{curve_type}_fits1.ecsv", format='ascii.ecsv', overwrite=True)
    table_curves2.write(f"../Data/SN{curve_type}_fits2.ecsv", format='ascii.ecsv', overwrite=True)
else:
    table_curves.write(f"../Data/SN{curve_type}_fits.ecsv", format='ascii.ecsv', overwrite=True)

/Users/tylerdonahue/miniforge3/envs/carpentries/lib/python3.14/site-packages/george/kernels.py:94: RuntimeWarning: divide by zero encountered in log
  log_constant = np.log(float(b)/self.ndim)
